In [ ]:
import os
os.environ["JAX_PLATFORM_NAME"] = "cpu"
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"

import jax
import jax.numpy as jnp
import jax.random as jrandom
import numpy as np

from typing import NamedTuple

Array = jax.Array

Time homogeneous vs time inhomogeneous Markov Chains

In [ ]:
# timeout
T = 100

# number of upper-level transitions
K = 10

init_p = 0.5
init_base_transition = np.array([
    [init_p, 1.0 - init_p],
    [0.0, 1.0],
])

init_dist = np.array([1.0, 0.0])

In [ ]:
seed = 42
num_samples = 1000

rng = jrandom.PRNGKey(seed)

In [ ]:
class StepState(NamedTuple):
    curr_rng: jrandom.PRNGKey
    done_t: Array
    curr_s: Array
    curr_transition: Array

@jax.vmap(in_axes=[0, 0, None])
def sample_transition(rng, curr_s, transition):
    return jrandom.choice(
        rng,
        2,
        p=transition[curr_s],
    )

def simulate(rng, transition, init_dist, num_samples):
    @jax.jit
    def step(step_i, step_state):
        curr_s = step_state.curr_s
        curr_transition = step_state.curr_transition

        curr_transition = jax.lax.select(
            (step_i + 1) % T == 0,
            curr_transition.at[0].set(
                curr_transition[0] / 2
            ),
            curr_transition,
        )

        done_t = step_state.done_t
        done_t = jax.lax.select(
            jnp.logical_and(
                curr_s == 1,
                jnp.logical_not(jnp.isfinite(done_t)),
            ),
            jnp.full_like(done_t, fill_value=step_i),
            done_t,
        )

        curr_rng = jrandom.fold_in(
            step_state.curr_rng,
            step_i,
        )
        curr_rng = jrandom.split(curr_rng, num_samples)

        next_s = sample_transition(
            curr_rng,
            curr_s,
            curr_transition,
        )

        return StepState(
            curr_rng=rng,
            done_t=done_t,
            curr_s=next_s,
            curr_transition=curr_transition,
        )

    init_state = jrandom.choice(
        rng,
        2,
        p=init_dist,
        shape=(num_samples,),
    )

    return jax.lax.fori_loop(
        0,
        T,
        step,
        StepState(
            curr_rng=rng,
            done_t=np.full(num_samples, fill_value=jnp.nan),
            curr_s=init_state,
            curr_transition=transition,
        )
    )


In [ ]:
p_range = np.arange(0.0, 1.0, 0.05)

In [ ]:
all_res = {}

for curr_p in p_range:
    curr_transition = np.array([
        [curr_p, 1.0 - curr_p],
        [0.0, 1.0],
    ])
    all_res[curr_p] = simulate(
        rng,
        curr_transition,
        init_dist,
        num_samples,
    )

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.plot(
    p_range,
    [
        jnp.nanmean(all_res[curr_p].done_t)
        for curr_p in p_range
    ]
)